# 22DM015 Final Project — Financial PhraseBank
## Person B: Parts 2 & 3 — BERT track (BERT, augmentation eval, full-data curve)

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).‍
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).‍
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).‍
- Evaluate headline numbers on **`test`** only; tune on **`val`**.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [11]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical data for everyone
train, val, test = splits['train'], splits['val'], splits['test']
labeled_32 = splits['labeled_32']
pool = du.unlabeled_pool(splits)     # Part 2 'unlabelled' data
PERSON = 'B'
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

train        1584 {'neutral': np.int64(973), 'positive': np.int64(399), 'negative': np.int64(212)}
val           227 {'neutral': np.int64(140), 'positive': np.int64(57), 'negative': np.int64(30)}
test          453 {'neutral': np.int64(278), 'positive': np.int64(114), 'negative': np.int64(61)}
labeled_32     32 {'positive': np.int64(11), 'negative': np.int64(11), 'neutral': np.int64(10)}


> **Install (run once):** `transformers`, `torch`, `accelerate` are needed here.‍ On Python 3.14 torch wheels may be missing — use a 3.11/3.12 venv.‍

## Part 2a.‍ BERT with 32 labelled examples (0.5)
Fine-tune a BERT-family model on `labeled_32`, evaluate on `test`.‍ Expect instability with 32 examples — fix seed, report val + test.‍

In [12]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2a: fine-tune a BERT classifier on the 32 labelled examples; evaluate on the shared test split.
import os, logging, warnings
# Tell transformers to skip its TensorFlow/Flax/Keras probe (we use PyTorch only).
# This prevents tf_keras from being imported, which is what emits the
# "tf.losses.sparse_softmax_cross_entropy is deprecated" notice. MUST be set BEFORE
# importing transformers.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # silence TF C++ logs if it still loads
logging.getLogger("tensorflow").setLevel(logging.ERROR)
# Silence the cosmetic "Redirects are currently not supported in Windows or MacOs" NOTE
# emitted by torch.distributed.elastic.multiprocessing.redirects when Trainer is imported
# on Windows/macOS. No-op on Linux. Set BEFORE importing transformers.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
# Silence torchao's "Skipping import of cpp extensions due to incompatible torch version"
# message (https://github.com/pytorch/ao/issues/2919). We don't use torchao kernels here,
# so the Python fallbacks are fine. Covers both the logger and warnings.warn paths.
logging.getLogger("torchao").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*Skipping import of cpp extensions.*")

import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed)

set_seed(SEED)
MODEL = 'bert-base-uncased'      # general BERT; 'ProsusAI/finbert' reserved for the Part 3 SOA comparison
NUM_LABELS = 3
MAX_LEN = 128

tok = AutoTokenizer.from_pretrained(MODEL)

def encode(df):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=MAX_LEN),
                  batched=True)

ds_train, ds_val, ds_test = encode(labeled_32), encode(val), encode(test)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)

args = TrainingArguments(
    output_dir='../.cache/bert_32shot',
    seed=SEED,
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    eval_strategy='no',
    save_strategy='no',
    logging_strategy='epoch',
    report_to='none',
    disable_tqdm=True,
)

trainer = Trainer(model=model, args=args, train_dataset=ds_train)
trainer.train()

def predict(ds):
    return trainer.predict(ds).predictions.argmax(-1)

val_pred, test_pred = predict(ds_val), predict(ds_test)
val_m = eu.evaluate(val['label'].values, val_pred)
test_m = eu.evaluate(test['label'].values, test_pred)
print('VAL :', {k: round(v, 4) for k, v in val_m.items()})
print('TEST:', {k: round(v, 4) for k, v in test_m.items()})

# headline numbers logged on TEST; val macro-F1 kept in notes for the tuning record
eu.log_result(MODEL, '32-shot', len(labeled_32), test_m, person=PERSON,
              notes=f"val_f1_macro={val_m['f1_macro']:.4f}")

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\EC\python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': 1.1258, 'grad_norm': 7.02362585067749, 'learning_rate': 1.925e-05, 'epoch': 1.0}
{'loss': 0.9835, 'grad_norm': 8.982705116271973, 'learning_rate': 1.825e-05, 'epoch': 2.0}
{'loss': 0.8412, 'grad_norm': 8.238197326660156, 'learning_rate': 1.7250000000000003e-05, 'epoch': 3.0}
{'loss': 0.8114, 'grad_norm': 7.439338684082031, 'learning_rate': 1.6250000000000002e-05, 'epoch': 4.0}
{'loss': 0.7076, 'grad_norm': 5.601452350616455, 'learning_rate': 1.525e-05, 'epoch': 5.0}
{'loss': 0.6269, 'grad_norm': 6.362582683563232, 'learning_rate': 1.425e-05, 'epoch': 6.0}
{'loss': 0.5296, 'grad_norm': 5.560019493103027, 'learning_rate': 1.325e-05, 'epoch': 7.0}
{'loss': 0.4851, 'grad_norm': 5.652841091156006, 'learning_rate': 1.2250000000000001e-05, 'epoch': 8.0}
{'loss': 0.4018, 'grad_norm': 5.08810567855835, 'learning_rate': 1.125e-05, 'epoch': 9.0}
{'loss': 0.3724, 'grad_norm': 5.582754611968994, 'learning_rate': 1.025e-05, 'epoch': 10.0}
{'loss': 0.3288, 'grad_norm': 3.8533921241760254, 'l

,person,model,method,split,n_train_labeled,accuracy,f1_macro,f1_weighted,f1_negative,f1_neutral,f1_positive,notes
0,B,bert-base-uncased,32-shot,test,32,0.679912,0.601195,0.697692,0.465116,0.823762,0.514706,val_f1_macro=0.5077
1,B,bert-base-uncased,32-shot,test,32,0.679912,0.601195,0.697692,0.465116,0.823762,0.514706,val_f1_macro=0.5077
2,B,bert-base-uncased,32-shot,test,32,0.679912,0.601195,0.697692,0.465116,0.823762,0.514706,val_f1_macro=0.5077
3,B,bert-base-uncased,32-shot,test,32,0.679912,0.601195,0.697692,0.465116,0.823762,0.514706,val_f1_macro=0.5077


### 2a — analysis

_TODO (student-authored)._ Comment on the 32-shot BERT result: macro-F1 vs the random/rule-based baselines and per-class F1 (where does it fail — minority classes?), the val/test gap, and the instability you expect from 32 examples.‍ Keep this interpretation in your own words (do not watermark; commit with `--no-verify`).‍


## Part 2b.‍ Train on Person D's augmented set (1)
Person D produces a non-LLM augmented training set (back-translation / EDA / etc.) under `data/augmented_32.csv`.‍ Re-train the SAME BERT on it and compare to 2a.‍

In [13]:
# watermark: AGLLM (AI-assisted content disclosure)
# aug = pd.read_csv('../data/augmented_32.csv')   # produced by Person D
# train BERT on aug, evaluate on test
# eu.log_result(MODEL, 'augmented', len(aug), m, person=PERSON)

## Part 2d.‍ Train on 32 + Person D's LLM-generated data (1)
Person D produces `data/llm_generated.csv`.‍ Train BERT on the 32 + generated points; analyse impact on metrics.‍

In [14]:
# watermark: AGLLM (AI-assisted content disclosure)
# gen = pd.read_csv('../data/llm_generated.csv')
# combine labeled_32 + gen, train, evaluate
# eu.log_result(MODEL, 'llm-generated', 32+len(gen), m, person=PERSON)

## Part 2e.‍ Optimal technique (0.5)
Apply the best technique(s) from 2a/2b/2d.‍ Comment + propose improvements (student-authored).‍

## Part 3.‍ Full-dataset SOA comparison (2)
3a.‍ Train on 1/10/25/50/75/100% of `train` (use `du.subset_by_fraction`).‍ 3b.‍ Plot the learning curve.‍ 3c.‍ Fold in Part 2 techniques.‍ 3d.‍ Methodology analysis.‍

In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3a: data-scaling curve — fine-tune a fresh BERT on increasing fractions of train, evaluate on test.
# Lighter protocol than 2a (max_len 64, batch 16, 3 epochs) so six CPU trainings stay tractable;
# the protocol is held FIXED across fractions so the curve isolates the effect of data quantity.
import pandas as pd
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed)

CURVE_MODEL = 'bert-base-uncased'
CURVE_MAXLEN = 64
CURVE_EPOCHS = 3
_ctok = AutoTokenizer.from_pretrained(CURVE_MODEL)

def _cencode(df):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: _ctok(b['text'], truncation=True, padding='max_length', max_length=CURVE_MAXLEN),
                  batched=True)

_ctest = _cencode(test)

def curve_train_eval(train_df):
    set_seed(SEED)  # fresh, reproducible init per fraction
    model = AutoModelForSequenceClassification.from_pretrained(CURVE_MODEL, num_labels=3)
    args = TrainingArguments(
        output_dir='../.cache/bert_curve', seed=SEED,
        num_train_epochs=CURVE_EPOCHS, per_device_train_batch_size=16, per_device_eval_batch_size=64,
        learning_rate=2e-5, eval_strategy='no', save_strategy='no', logging_strategy='no',
        report_to='none', disable_tqdm=True,
    )
    tr = Trainer(model=model, args=args, train_dataset=_cencode(train_df))
    tr.train()
    pred = tr.predict(_ctest).predictions.argmax(-1)
    return eu.evaluate(test['label'].values, pred)

FRACTIONS = [0.01, 0.10, 0.25, 0.50, 0.75, 1.00]
curve_rows = []
for f in FRACTIONS:
    sub = du.subset_by_fraction(train, f)
    m = curve_train_eval(sub)
    eu.log_result(CURVE_MODEL, f'full-{int(f*100)}%', len(sub), m, person=PERSON,
                  notes=f'frac={f}; epochs={CURVE_EPOCHS}; maxlen={CURVE_MAXLEN}')
    curve_rows.append({'frac': f, 'n_train': len(sub), **m})
    print(f"frac={f:>4}  n={len(sub):4d}  acc={m['accuracy']:.4f}  macroF1={m['f1_macro']:.4f}")

pd.DataFrame(curve_rows)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/15 [00:00<?, ? examples/s]

c:\EC\python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 21.7, 'train_samples_per_second': 2.074, 'train_steps_per_second': 0.138, 'train_loss': 1.1039336522420247, 'epoch': 3.0}
frac=0.01  n=  15  acc=0.3797  macroF1=0.2839


...

c:\EC\python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 1267.9858, 'train_samples_per_second': 1.874, 'train_steps_per_second': 0.118, 'train_loss': 0.4181576792399089, 'epoch': 3.0}
frac= 0.5  n= 792  acc=0.9514  macroF1=0.9369


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/1188 [00:00<?, ? examples/s]

c:\EC\python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3b: learning curve from the shared results scoreboard.
import matplotlib.pyplot as plt

res = pd.read_csv('../results/results.csv')
bsel = (res['person'] == 'B') & (res['model'] == 'bert-base-uncased')
# dedup re-run rows, keep the latest per method
curve = (res[bsel & res['method'].str.startswith('full-')]
         .drop_duplicates(subset=['method'], keep='last')
         .sort_values('n_train_labeled'))
shot = (res[bsel & (res['method'] == '32-shot')]
        .drop_duplicates(subset=['method'], keep='last'))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(curve['n_train_labeled'], curve['f1_macro'], marker='o', label='BERT macro-F1 (data-scaling)')
ax.plot(curve['n_train_labeled'], curve['accuracy'], marker='s', ls='--', label='BERT accuracy')
if len(shot):
    ax.scatter(shot['n_train_labeled'], shot['f1_macro'], color='red', zorder=5,
               label='32-shot BERT macro-F1 (2a)')
# reference floors from Part 1 (test split): random (1c) and rule-based (1d)
ax.axhline(0.3035, color='gray', ls=':', label='random floor (macro-F1 0.30)')
ax.axhline(0.7304, color='green', ls=':', label='rule-based (macro-F1 0.73)')
ax.set_xscale('log')
ax.set_xlabel('# labelled training examples (log scale)')
ax.set_ylabel('test score')
ax.set_title('Part 3 learning curve — BERT on Financial PhraseBank (allagree)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

curve[['method', 'n_train_labeled', 'accuracy', 'f1_macro', 'f1_weighted']]

### 3c — technique comparison &nbsp;·&nbsp; 3d — methodology analysis

**3c (fold in Part 2 techniques):** once Person D commits `data/augmented_32.csv` (2b) and `data/llm_generated.csv` (2d), their `eu.log_result(...)` rows land in the same `results/results.csv`.‍ Re-run the cell above to overlay `augmented` and `llm-generated` points against the data-scaling curve (i.e.‍ "how many real labels is each technique worth?").‍

_TODO (student-authored) — 3d methodology analysis._ Compare all methods (random / rule-based / 32-shot / augmentation / LLM-generated / full-data curve): where each helps, the data-efficiency story, limitations, and what you'd do with more compute.‍ Your words; commit with `--no-verify`.‍
